In [ ]:
import json
from collections import Counter
import pandas as pd
import os
import requests
from rdflib import Graph, URIRef, Literal, Namespace, ConjunctiveGraph, Dataset

In [ ]:
csv_pfad ="rulemapped_and_cleaned_findings.csv"
df = pd.read_csv(csv_pfad)

In [ ]:
df_double_findings = df["filename"].value_counts().sort_values(ascending=False)
print(df_double_findings)

In [ ]:
df_double_findings = (df["filename"].value_counts().sort_values(ascending=False).reset_index())
df_double_findings.columns = ["filename", "count"]

In [ ]:
unique_ids = df["found_cwe_id"].unique()
print(unique_ids)

In [ ]:
unique_gt_cwe = df["groundtruth_cwe_id"].unique()
print(unique_gt_cwe)

In [ ]:
ds  = Dataset()
ds.parse("named_graphs_cwe.trig", format="trig")

In [ ]:
def get_children_ontov(cwe_id, ontology):
    children = [q[0].split("/")[-1] for q in ontology.quads((None, URIRef("http://schema.org/child_of"), URIRef(f"https://cwe.mitre.org/data/definitions/{cwe_id}"), None))]
    new_c = []
    for child in children:
        children_new = get_children_ontov(child, ontology)
        new_c.extend(children_new)
    children.extend(new_c)
    return children


In [ ]:
def get_parents_ontov(cwe_id, ontology):
    parents = [q[2].split("/")[-1] for q in ontology.quads((URIRef(f"https://cwe.mitre.org/data/definitions/{cwe_id}"), URIRef("http://schema.org/child_of"), None, None))]    
    new_p = []
    for parent in parents:
        parents_new = get_parents_ontov(parent, ontology)
        new_p.extend(parents_new)
    parents.extend(new_p)
    return parents
    

In [ ]:
def safe_get_relatives(cwe_id, func, dataset):
    if pd.isna(cwe_id) or not isinstance(cwe_id, str) or cwe_id.strip() == "":
        return []
    try:
        return list(set(func(cwe_id.lower(), dataset)))
        
    except Exception as e:
        print(f"error for {cwe_id}: {e}")
        return []
        

In [ ]:
def cleaned_cwe_id(cwe):
    if pd.isna(cwe) or not isinstance(cwe, str):
        return None
    return cwe.strip().lower().replace("cwe-", "")


In [ ]:
df["cwe_id_clean"] = df["found_cwe_id"].apply(cleaned_cwe_id)
df["cwe_gt_clean"] = df["groundtruth_cwe_id"].apply(cleaned_cwe_id)

In [ ]:
def format_cwe_list(cwe_list):
    if not isinstance(cwe_list, list):
        return []
    formatted = []
    for cwe in cwe_list:
        if cwe is None or str(cwe).strip() == "":
            continue
        cwe_str = str(cwe).strip()
        if cwe_str.startswith("cwe-"):
            formatted.append(cwe_str)
        else:
            formatted.append(f"cwe-{cwe_str}")
    return formatted
    

In [ ]:
def match_finding(row):
    gt = row["groundtruth_cwe_id"]
    found = row["cwe_id_clean"]
    related = row["related"]

    if pd.isna(gt) or gt.strip() == "":
        return "no_gt"

    gt_norm = gt.lower() if gt.lower().startswith("cwe-") else f"cwe-{gt.lower()}"
    found_norm = found if found and found.startswith("cwe-") else (f"cwe-{found.lower()}" if found else None)

    if found_norm == gt_norm:
        return "EP"
    elif found_norm is not None and gt_norm in related:
        return "HP"
    elif found_norm is not None:
        return "mismatch"
    else:
        return "no_finding"


In [ ]:
def compute_final_verdict(group):
    file_name = group["filename"].iloc[0]
    ep_count = (group["match_result"] == "EP").sum()
    hp_count = (group["match_result"] == "HP").sum()
    mismatch_count = (group["match_result"] == "mismatch").sum()

    if ep_count > 0:
        verdict = "EP"
    elif hp_count > 0:
        verdict = "HP"
    elif mismatch_count > 0:
        verdict = "mismatch"
    else:
        verdict = "no_finding"

    return pd.Series({
        "filename": file_name,
        "final_verdict": verdict,
        "hp_count": hp_count,
        "mismatch_count": mismatch_count,
        "ep_count": ep_count
    })


In [ ]:
df["sample_name"] = (df["file_path"].str.replace("code_before/", "", regex=False).str.replace("/", "_", regex=False).str.lower())

df["cwe_id_clean"] = df["found_cwe_id"].apply(cleaned_cwe_id)

df["children"] = df["cwe_id_clean"].apply(lambda cwe: safe_get_relatives(cwe, get_children_ontov, ds))
df["parents"] = df["cwe_id_clean"].apply(lambda cwe: safe_get_relatives(cwe, get_parents_ontov, ds))

df["children"] = df["children"].apply(format_cwe_list)
df["parent"] = df["parents"].apply(format_cwe_list)

df["related"] = df.apply(lambda row: row["children"] + row["parents"], axis=1)

In [ ]:
df["match_result"] = df.apply(match_finding, axis=1)

In [ ]:
sample_results = df.groupby("sample_name").apply(compute_final_verdict).reset_index(drop=True)

In [ ]:
df = df.merge(sample_results, on="filename", how="left")

In [ ]:
df.to_csv("scan_result_evaluation.csv", index=False)

In [ ]:
eps_sample = sample_results[sample_results["final_verdicT"] == "EP"]
print(f"num of samples with veridct ep: {len(eps_sample)}")

In [ ]:
hps_sample = sample_results[sample_results["final_verdict"] == "HP"]
print(f"num of samples with veridct hp: {len(hps_sample)}")

In [ ]:
mismatches_sample = sample_results[sample_results["final_verdict"] == "mismatch"]
print(f"num of samples with veridct mismatch: {len(mismatches_sample)}")

In [ ]:
df_match = df[df["match_result"] == "EP"]
print(len(df_match))

In [ ]:
df_hp_match = df[df["match_result"] == "HP"]
print(len(df_hp_match))

In [ ]:
df_hp_ep = pd.concat([df_match, df_hp_match])
print(len(df_hp_ep))

In [ ]:
df_mismatch = df[df["match_result"] == "mismatch"]
print(len(df_mismatch))